# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


In [3]:
content_preview = con.sql(f"""
    SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 5
""").df()
list(content_preview.columns)

['client_hash_id',
 'content_hash_id',
 'keyword_hash_id',
 'url_hash_id',
 'keyword_char_count',
 'keyword_token_count',
 'url_char_count',
 'content_created_date',
 'content_updated_date',
 'content_type',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'main_intent',
 'backlinks',
 'category_count',
 'keyword_created_date',
 'provider_used',
 'model_used',
 'char_count',
 'word_count',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published',
 'is_deleted']

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule (plain words):**
A content page is worth reviewing if it hasn't been updated in a long time (stale),
but it's still getting meaningful search visibility (visible) — this combination
signals a real opportunity to refresh and recover lost performance.

**Reason codes this rule can output:**
- `stale_but_visible` — old content that still gets impressions; a refresh candidate.
- `not_flagged` — content that's either fresh or has too little visibility to matter.
**Note on staleness calculation:** Since `content_updated_date` reflects the latest
snapshot (many rows show updates after March 2026), I treat any content not yet
updated by March 31, 2026 as "stale as of March" — this avoids using future update
dates as if they were known at decision time.

In [6]:
signal1 = con.sql(f"""
    WITH content_perf AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(f.gsc_impressions) AS impressions,
            SUM(f.gsc_clicks) AS clicks
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
        WHERE f.gsc_data_available IS TRUE
        GROUP BY f.client_hash_id, f.content_hash_id
    ),
    joined AS (
        SELECT
            cp.*,
            CASE WHEN c.content_updated_date > DATE '2026-03-31' OR c.content_updated_date IS NULL
                 THEN 'stale (not updated by March)'
                 ELSE 'fresh (updated by March)'
            END AS bucket
        FROM content_perf cp
        JOIN read_parquet('{BASE}/dim_content.parquet') c
          ON cp.content_hash_id = c.content_hash_id AND cp.client_hash_id = c.client_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
    )
    SELECT
        bucket,
        COUNT(*) AS n,
        AVG(clicks) AS avg_clicks,
        AVG(impressions) AS avg_impressions
    FROM joined
    GROUP BY bucket
""").df()
signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n,avg_clicks,avg_impressions
0,fresh (updated by March),27801,2.807345,1239.622963
1,stale (not updated by March),148767,4.998622,1654.639524


**Signal 1 — Staleness — Verdict: OPPOSITE**

Stale content (n=148,767) averaged 4.99 clicks and 1,655 impressions, while fresh
content (n=27,801) averaged only 2.81 clicks and 1,240 impressions. This is the
opposite of what my rule assumed — stale content is NOT underperforming. A likely
explanation: content left unupdated may be older, established, high-traffic pages
that don't need updates, while "fresh" content may be newer and still ramping up.
This finding changes my rule (see below).

In [7]:
signal2 = con.sql(f"""
    WITH content_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        CASE WHEN avg_position <= 10 THEN 'good_position (top 10)' ELSE 'poor_position (below 10)' END AS bucket,
        COUNT(*) AS n,
        AVG(clicks * 1.0 / impressions) AS avg_ctr
    FROM content_perf
    GROUP BY bucket
""").df()
signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n,avg_ctr
0,good_position (top 10),55895,0.003294
1,poor_position (below 10),45546,0.001784


**Signal 2 — CTR vs Position — Verdict: CONFIRMED**

Top-10 position content (n=55,895) averaged 0.33% CTR, compared to 0.18% CTR for
content ranked below 10 (n=45,546) — roughly double, in the expected direction
(better position → higher CTR). This confirms position and CTR are related as the
CTR-fix flag assumes. Both averages are low overall, meaning individual pages
with good position but unusually low CTR (below their bucket average) are the
real opportunity — this is what my rule will target.

**Updated rule (based on signal testing):**
Since Signal 1 showed staleness does NOT predict weak performance (opposite of
assumption), I drop staleness from the rule. Instead: a content page is worth
reviewing if it holds a good search position (top 10) but its CTR is unusually
low compared to other top-10 pages — a likely title/snippet problem, not a
ranking problem.

**Reason codes:**
- `ctr_fix_opportunity` — good position, below-average CTR for its bucket.
- `not_flagged` — either poor position or CTR already at/above bucket average.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This section encodes the rule confirmed in Section 1: flag content with good
search position (top 10) but CTR below its bucket's average — a likely
title/snippet fix opportunity, not a ranking problem.

In [8]:
import os

# Step 1: aggregate March performance per content item
queue = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
""").df()

queue["ctr"] = queue["clicks"] / queue["impressions"]

# Step 2: define good_position (transparent, no fitted weights)
queue["good_position"] = (queue["avg_position"] <= 10).astype(int)

# Step 3: bucket average CTR for top-10 content only
bucket_avg_ctr = queue.loc[queue["good_position"] == 1, "ctr"].mean()

# Step 4: the rule — score + reason code + action label
queue["low_ctr_flag"] = ((queue["good_position"] == 1) & (queue["ctr"] < bucket_avg_ctr)).astype(int)

queue["score"] = queue["low_ctr_flag"] * queue["impressions"]  # readable: bigger impressions = bigger opportunity

queue["reason_code"] = queue["low_ctr_flag"].map({1: "ctr_fix_opportunity", 0: "not_flagged"})
queue["action"] = queue["low_ctr_flag"].map({1: "review_title_and_snippet", 0: "no_action"})

# Step 5: rank
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

# Step 6: write CSV
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(queue))
print("Flagged (ctr_fix_opportunity):", queue["low_ctr_flag"].sum())
queue.head(10)

Rows written: 101441
Flagged (ctr_fix_opportunity): 36331


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,good_position,low_ctr_flag,score,reason_code,action,rank
0,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,1,1,221310.0,ctr_fix_opportunity,review_title_and_snippet,1
1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,1,1,212404.0,ctr_fix_opportunity,review_title_and_snippet,2
2,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,1,1,203497.0,ctr_fix_opportunity,review_title_and_snippet,3
3,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.450106,0.001858,1,1,194337.0,ctr_fix_opportunity,review_title_and_snippet,4
4,client_e547b89c05043229,content_4ffe18112a5642e3,186983.0,586.0,2.331060,0.003134,1,1,186983.0,ctr_fix_opportunity,review_title_and_snippet,5
5,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.361195,0.001534,1,1,170808.0,ctr_fix_opportunity,review_title_and_snippet,6
6,client_73cda7b4e4f265ea,content_471d9cabce329a66,164885.0,396.0,4.656030,0.002402,1,1,164885.0,ctr_fix_opportunity,review_title_and_snippet,7
7,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,151166.0,408.0,3.391428,0.002699,1,1,151166.0,ctr_fix_opportunity,review_title_and_snippet,8
8,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,1,1,143019.0,ctr_fix_opportunity,review_title_and_snippet,9
9,client_73cda7b4e4f265ea,content_e241d6415ac9e534,142304.0,343.0,3.276016,0.002410,1,1,142304.0,ctr_fix_opportunity,review_title_and_snippet,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = queue.head(20)[["rank", "content_hash_id", "impressions", "clicks", "avg_position", "ctr", "reason_code", "action"]]
top20


,rank,content_hash_id,impressions,clicks,avg_position,ctr,reason_code,action
0,1,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,ctr_fix_opportunity,review_title_and_snippet
1,2,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,ctr_fix_opportunity,review_title_and_snippet
2,3,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,ctr_fix_opportunity,review_title_and_snippet
3,4,content_b99ea6861864dea5,194337.0,361.0,4.450106,0.001858,ctr_fix_opportunity,review_title_and_snippet
4,5,content_4ffe18112a5642e3,186983.0,586.0,2.331060,0.003134,ctr_fix_opportunity,review_title_and_snippet
5,6,content_acbcc847f8996314,170808.0,262.0,3.361195,0.001534,ctr_fix_opportunity,review_title_and_snippet
6,7,content_471d9cabce329a66,164885.0,396.0,4.656030,0.002402,ctr_fix_opportunity,review_title_and_snippet
7,8,content_fd2117c2c6790e4b,151166.0,408.0,3.391428,0.002699,ctr_fix_opportunity,review_title_and_snippet
8,9,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,ctr_fix_opportunity,review_title_and_snippet
9,10,content_e241d6415ac9e534,142304.0,343.0,3.276016,0.002410,ctr_fix_opportunity,review_title_and_snippet


## Top-20 Review

All 20 rows share `reason_code = ctr_fix_opportunity` and `action = review_title_and_snippet`.

| Rank | Content ID | Clicks / Impr. | CTR | Confidence | What would make it wrong? |
|---|---|---|---|---|---|
| 1 | content_0e03de76... | 720 / 221K | 0.33% | **High** — large click volume, stable estimate | Title was recently changed; this may already be stale data |
| 2 | content_44f34c0a... | 24 / 212K | 0.01% | Medium — striking CTR, but thin click count | Query is low-intent by nature, not a snippet problem |
| 3 | content_8d7d99f1... | 289 / 203K | 0.14% | **High** — solid click volume | SERP features (snippet/ads) may absorb clicks regardless of title |
| 4 | content_b99ea686... | 361 / 194K | 0.19% | **High** | Seasonal demand dip, not a title issue |
| 5 | content_4ffe1811... | 586 / 187K | 0.31% | **High** | Already near-normal CTR for its category |
| 6 | content_acbcc847... | 262 / 171K | 0.15% | Medium-High | Position (3.4) fluctuates daily; average may hide good days |
| 7 | content_471d9cab... | 396 / 165K | 0.24% | **High** | Competitor snippet recently outranked — positioning, not title issue |
| 8 | content_fd2117c2... | 408 / 151K | 0.27% | **High** | Informational intent — users rarely click any result |
| 9 | content_34a70fea... | 43 / 143K | 0.03% | Medium — low clicks vs. impressions is suspicious | Possible tracking/attribution anomaly |
| 10 | content_e241d641... | 343 / 142K | 0.24% | **High** | Recent SERP layout change for this query |
| 11 | content_f43118e0... | 191 / 139K | 0.14% | Medium-High | Position (5.0) is borderline; small rank shifts explain CTR |
| 12 | content_f352b7cf... | 287 / 136K | 0.21% | **High** | Branded/navigational query — low CTR is expected here |
| 13 | content_8e1334d6... | **1** / 135K | 0.0007% | **Low** — single click is noise, not signal | Likely a data artifact — **weak pick** |
| 14 | content_7c637314... | 83 / 133K | 0.06% | Medium — thin click count | Position (5.8) may be worse on some days than the average shows |
| 15 | content_fe3fd342... | 371 / 125K |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

Two picks from the top-20 are weak: **#13 (content_8e1334d6...)** and
**#16 (content_fec55986...)** — both have only 1 click against 120K+ impressions.
A CTR built on a single click is statistical noise, not a reliable pattern; these
should not be trusted as strong `ctr_fix_opportunity` flags despite their high
score. A rule refinement worth considering: require a minimum click count
(e.g. clicks >= 10) alongside the CTR condition, so the score isn't dominated by
impressions alone when clicks are near-zero.

## Leakage check

I confirm this rule uses only data observed by the end of March 2026:
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — all trailing March
  performance, no future window.
- No product/action flags were used (`last_optimized_date` and
  `optimization_eligible_date` from `dim_content` were deliberately excluded —
  these indicate FlyRank's own past actions on the content, which would leak
  "this content was already flagged" information into the score).
- No label-derived columns are present — this is a rule, not a trained model, so
  there is no label to leak from.

In [10]:
# Confirm the rule's columns don't include any product/action flags
rule_columns_used = ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
excluded_flags = ["last_optimized_date", "optimization_eligible_date"]

print("Columns used in rule:", rule_columns_used)
print("Columns deliberately excluded (product flags):", excluded_flags)
print("Any overlap?", set(rule_columns_used) & set(excluded_flags))


Columns used in rule: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Columns deliberately excluded (product flags): ['last_optimized_date', 'optimization_eligible_date']
Any overlap? set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.